In [7]:
1+1

2

In [8]:
from langchain_core.documents import Document
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\KEERTHI VARDHAN\AppData\Local\Temp\ipykernel_23596\2611768676.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
c:\Basicrag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### RAG PIPELINE

In [9]:
### read all the documents in the directory
def read_all_documents(pdf_directory):
    """Read all the documents from the pdf"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"{len(pdf_files)}to process")
    for pdf_file in pdf_files:
        print(f"processing{pdf_file.name}")
        try:
            Loader = PyPDFLoader(str(pdf_file))
            documents = Loader.load()
            ## add source to the metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
            all_documents.extend(documents)
            print(f"loaded {len(documents)} pages")
        except Exception as e:
            print(f"Error processing  {e}")
            
    print(f"the number of {len(all_documents)}")
    return all_documents
all_the_documents = read_all_documents("../data")
        


3to process
processing1-4 Deep learnng.pdf
loaded 5 pages
processing5-8 Deep LEarning.pdf
loaded 6 pages
processingKeerthi_Vardhan_Genpact_DataScientist.pdf
loaded 2 pages
the number of 13


In [10]:
all_the_documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\1-4 Deep learnng.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': '1-4 Deep learnng.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\1-4 Deep learnng.pdf', 'total_pages': 5, 'page': 1, 'page_label': '2', 'source_file': '1-4 Deep learnng.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\1-4 Deep learnng.pdf', 'total_pages': 5, 'page': 2, 'page_label': '3', 'source_file': '1-4 Deep learnng.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\1-4 Deep learnng.pdf', 'total_pages': 5, 'page': 3, 'page_label': '4', 'source_file': '1-4 Deep learnng.pdf', 'file_type': 'p

In [11]:
type(all_the_documents[0])

langchain_core.documents.base.Document

In [12]:
## TextSplitter to split the documents into smalle chunks
def split_documnets(documents,chunk_size = 1000,chunk_overlap=200):
    """split doduments into smaller chunks for better RAG performance"""
    split_doc = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    splitter_text = split_doc.split_documents(documents)
    print(f"split {len(documents)} into {len(splitter_text)} chunks")
    
    ## Show example of a chunk
    if splitter_text:
        print(f"Example of a chunk: {splitter_text[0].page_content[:200]}...")
        print(f"Metadata of the chunk: {splitter_text[0].metadata}")
        
    return splitter_text

In [13]:
chunk = split_documnets(all_the_documents)
chunk

split 13 into 6 chunks
Example of a chunk: KEERTHI VARDHAN NAIDU NETTEM
GitHub | LinkedIn | nk.vardhannaidu@gmail.com | +91-9949902603
PROFESSIONAL SUMMARY
Aspiring Data Scientist with hands-on experience in Artificial Intelligence, Machine Le...
Metadata of the chunk: {'producer': '', 'creator': 'WPS Docs', 'creationdate': '2026-06-14T10:13:35+05:30', 'author': 'Un-named', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-06-14T10:13:35+05:30', 'sourcemodified': "D:20260614101335+05'30'", 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Keerthi_Vardhan_Genpact_DataScientist.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Keerthi_Vardhan_Genpact_DataScientist.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': '', 'creator': 'WPS Docs', 'creationdate': '2026-06-14T10:13:35+05:30', 'author': 'Un-named', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-06-14T10:13:35+05:30', 'sourcemodified': "D:20260614101335+05'30'", 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Keerthi_Vardhan_Genpact_DataScientist.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Keerthi_Vardhan_Genpact_DataScientist.pdf', 'file_type': 'pdf'}, page_content='KEERTHI VARDHAN NAIDU NETTEM\nGitHub | LinkedIn | nk.vardhannaidu@gmail.com | +91-9949902603\nPROFESSIONAL SUMMARY\nAspiring Data Scientist with hands-on experience in Artificial Intelligence, Machine Learning, Predictive Analytics, and\nAdvanced Analytics. Skilled in developing and deploying end-to-end AI/ML solutions, performing exploratory data analysis,\nand delivering actionable recommendations from complex datasets. Experienced in collaborating with cross-functional\nte

### Embeddings and vector store db

In [14]:
import os
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
import numpy as np

In [16]:
class embeddingmanager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self,model_name:str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager
        
        Args:
            model_name:HuggingFace model name for sentence embedding
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model:{self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded Sucessfully .Embedding Dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise
        
    def generate_embeddings(self,texts :List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
        Args:
            texts: List of strings to be embedded
        Returns:
            np.ndarray: Array of embeddings"""
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    ## intialize the embedding manager
embedding_manager = embeddingmanager()
embedding_manager
        

Loading embedding model:all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3718.90it/s]


Model Loaded Sucessfully .Embedding Dimension: 384


### VectorStore

In [17]:
class VectorStore:
    """Manages documnent embeddings in a chromadb vector store"""
    
    def __init__(self,collection_name:str = "pdf_documents" ,persist_directory:str = "../data/vector_store"):
        """initilaize the vector store
        
        Args:
            collenction_name: Name of the chromadb collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize the chromadb client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description":"PDF document embeddings for RAG system"}
            )
            print(f"Vector Stote initialized with collection: {self.collection_name}")
            print(f"Existing documents in the collection: {self.collection.count()}")
        except  Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self,documents :List[Any],embeddings:np.ndarray):
        """Add documents and their embeddings to the vector store
        
        Args:
            documents: List of Langchain Documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents and embeddings must match")
        print(f"Adding {len(documents)} documents to the vector store...")
        
        # Prepare data for chromadb
        ids = []
        metadatas = []
        document_text = []
        embeddings_list = []
        
        for i,(doc,embedding )  in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['contact_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            #Document text
            
            document_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        # Addd to collection
        try:
            self.collection.add(
                ids = ids,
                metadatas = metadatas,
                documents = document_text,
                embeddings = embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in the collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
vector_store = VectorStore()
vector_store
            

Vector Stote initialized with collection: pdf_documents
Existing documents in the collection: 6


In [18]:
# Initialize
embedding_manager = embeddingmanager()
vector_store = VectorStore()

# Generate embeddings
texts = [doc.page_content for doc in chunk]
embeddings = embedding_manager.generate_embeddings(texts)

# Add to vector store
vector_store.add_documents(chunk, embeddings)

# Verify ✅
print(vector_store.collection.count())  # Now shows correct count!

Loading embedding model:all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3232.47it/s]


Model Loaded Sucessfully .Embedding Dimension: 384
Vector Stote initialized with collection: pdf_documents
Existing documents in the collection: 6
Generating embeddings for 6 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.11it/s]

Generated embeddings with shape: (6, 384)
Adding 6 documents to the vector store...
Successfully added 6 documents to the vector store.
Total documents in the collection: 12
12


In [41]:
print(vector_store.collection.count())

6


In [40]:
print(vector_store.collection.get())

{'ids': ['doc_f12aed3b_0', 'doc_9a4e2728_1', 'doc_ecc54d89_2', 'doc_b9d72e7e_3', 'doc_796d25b3_4', 'doc_99551a34_5'], 'embeddings': None, 'documents': ['KEERTHI VARDHAN NAIDU NETTEM\nGitHub | LinkedIn | nk.vardhannaidu@gmail.com | +91-9949902603\nPROFESSIONAL SUMMARY\nAspiring Data Scientist with hands-on experience in Artificial Intelligence, Machine Learning, Predictive Analytics, and\nAdvanced Analytics. Skilled in developing and deploying end-to-end AI/ML solutions, performing exploratory data analysis,\nand delivering actionable recommendations from complex datasets. Experienced in collaborating with cross-functional\nteams to understand data requirements and solve real-world business problems using data-driven approaches. Passionate\nabout staying up-to-date with the latest advancements in AI and data science.\nSKILLS\n• Programming Languages & Databases: Python, SQL (MySQL), HTML, CSS, JavaScript\n• AI & ML: Artificial Intelligence (AI), Machine Learning, Advanced Analytics, Pre

### Retrivel pipeline from vector store

In [32]:
class RAGretriver:
    """retrive the data"""
    def __init__(self,vector_store = VectorStore,embedding_manager = embeddingmanager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    def retrive_data(self,query:str,top_k:int= 5,score_threshold:float=0.0) -> List[Dict[str,Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retriving documents from the query {query}")
        print(f"Top k {top_k} and score_threshold{score_threshold}")
        
        ##Generate embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        ## search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )
            
            
            retrived_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    ## convert distance to similarity score
                    similarity_score = 1 - distance
                    if similarity_score > score_threshold:
                        retrived_docs.append({
                            "id":doc_id,
                            "content":document,
                            "metadata":metadata,
                            "similarity_score":similarity_score,
                            "distance":distance,
                            "rank":i+1
                        })
                        
                print(f"retrived {len(retrived_docs)}")
            else:
                print("No documents found")
            
            return retrived_docs
        
        except Exception as e:
            print(f"Error during retrivel {e}")
            return []
rag_retriver = RAGretriver(vector_store,embedding_manager)
                    
        
        

In [33]:
rag_retriver

In [34]:
rag_retriver.retrive_data("Developed an AI-powered ML model to predict telecom customer churn u")

Retriving documents from the query Developed an AI-powered ML model to predict telecom customer churn u
Top k 5 and score_threshold0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 28.68it/s]

Generated embeddings with shape: (1, 384)
retrived 4


[{'id': 'doc_ecc54d89_2',
  'content': '• Collected, cleaned, and preprocessed complex datasets for analysis; applied feature engineering and SMOTE-based\nclass balancing to improve model accuracy and robustness.\n• Conducted exploratory data analysis (EDA) using Matplotlib and Seaborn; interpreted results to provide actionable\nrecommendations to cross-functional teams.\n• Deployed AI/ML models as production-ready web applications using Flask and Streamlit; monitored system\nperformance and model evaluation metrics including Accuracy, F1-score, and ROC-AUC.\n• Applied cross-validation, hyperparameter tuning, and advanced analytics techniques to continuously improve model\nmethodologies and prediction accuracy.\nPROJECTS\nCustomer Churn Prediction GitHub\nTech Stack: Python, Streamlit, Scikit-learn, Pandas, NumPy, SMOTE, Advanced Analytics\n• Developed an AI-powered ML model to predict telecom customer churn using 7,000+ records; applied advanced\nanalytics to extract meaningful insigh

RAG PIPELINE-  vectordb to LLM output generation

In [ ]:
## simple rag pipline using groq llm
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = ""

llm = ChatGroq(groq_api_key = groq_api_key,model = "llama-3.3-70b-versatile")
llm

## Simple Rag function retrive data +generate response
def rag_sample(llm,query,retriever,top_k = 3):
    ## retrive the context
    results = retriever.retrive_data(query,top_k = top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else""
    if not context:
        return "No relavent context found to answer this question"
    prompt = f"""use the following context to answer the question consisly{context}
    
    question:{query}
    answer:"""
    response = llm.invoke([prompt.format(context = context,query = query)])
    return response.content
    


In [48]:
answer = rag_sample(llm,"Developed an AI-powered ML model to predict telecom customer churn",rag_retriver)
answer

Retriving documents from the query Developed an AI-powered ML model to predict telecom customer churn
Top k 3 and score_threshold0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 32.02it/s]

Generated embeddings with shape: (1, 384)
retrived 3


'Developed an AI-powered ML model to predict telecom customer churn using 7,000+ records; applied advanced analytics to extract meaningful insights and deliver actionable business recommendations. The model achieved 97% accuracy and 86% F1-score, and was deployed via Streamlit for real-time prediction with system performance monitoring using ROC-AUC, precision-recall, and confusion matrix.'